# I²-SDF: Intrinsic Indoor Scene Reconstruction with Physics-Guided Grounding
## Google Colab GPU Setup, Execution, and Evaluation Pipeline

This notebook provides an end-to-end, reproducible execution environment on Google Colab with CUDA GPU acceleration for **I²-SDF** and our **physics-guided grounding constraint**.

---
### Notebook Structure:
- **Section A**: GPU Verification (CUDA availability, VRAM memory, GPU specifications)
- **Section B**: Repository Setup & File Verification (Modified files integrity check)
- **Section C**: Dependency Installation (PyTorch Lightning 1.9.0, Open3D, PyMCubes, etc.)
- **Section D**: Dataset Setup (`bedroom_0` / `scan0` download & camera normalization)
- **Section E**: Environment & Modality Verification (OpenEXR, config loader, CUDA ops)
- **Section F**: Tiny Sanity Test (Optional 20-Iteration Check — NOT run automatically)
- **Section G**: Baseline Experiment (`ground_weight = 0.0` with safety confirmation guard)
- **Section H**: Physics Experiment (`ground_weight = 0.1` with safety confirmation guard)
- **Section I**: 3D Mesh Extraction (Marching Cubes Resolution 512)
- **Section J**: Quantitative Comparison & Grounding Metrics
- **Section K**: Persistent Storage & Google Drive Backup
- **Summary**: Exact CLI Command Reference


## Section A: GPU Verification
Verify that Google Colab is running with a CUDA GPU (e.g., T4, L4, V100, or A100) and query hardware specifications.

In [ ]:
# ==============================================================================
# SECTION A: GPU VERIFICATION
# ==============================================================================
import os
import sys
import torch

print("=" * 70)
print("             SECTION A: CUDA GPU HARDWARE VERIFICATION")
print("=" * 70)

cuda_available = torch.cuda.is_available()
print(f"[STATUS] torch.cuda.is_available() : {cuda_available}")

if not cuda_available:
    raise RuntimeError(
        "CUDA GPU is NOT available! In Google Colab, go to Runtime -> Change runtime type "
        "and select a GPU accelerator (T4, L4, or A100), then re-run this cell."
    )

gpu_count = torch.cuda.device_count()
gpu_name = torch.cuda.get_device_name(0)
gpu_capability = torch.cuda.get_device_capability(0)
total_memory_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
allocated_memory_gb = torch.cuda.memory_allocated(0) / (1024 ** 3)
cached_memory_gb = torch.cuda.memory_reserved(0) / (1024 ** 3)

print(f"[STATUS] Detected GPU Count        : {gpu_count}")
print(f"[STATUS] Primary GPU Device Name   : {gpu_name}")
print(f"[STATUS] Compute Capability        : {gpu_capability[0]}.{gpu_capability[1]}")
print(f"[STATUS] Total VRAM Available      : {total_memory_gb:.2f} GB")
print(f"[STATUS] Allocated VRAM            : {allocated_memory_gb:.2f} GB")
print(f"[STATUS] Reserved VRAM             : {cached_memory_gb:.2f} GB")
print(f"[STATUS] PyTorch Version           : {torch.__version__}")
print(f"[STATUS] CUDA Version in PyTorch   : {torch.version.cuda}")
print("=" * 70)
print("\n[nvidia-smi Output]:")
!nvidia-smi


## Section B: Repository Setup & File Integrity
Sets up the repository directory in Colab and verifies that the core implementation files and grounding constraint modifications are present.

In [ ]:
# ==============================================================================
# SECTION B: REPOSITORY SETUP & INTEGRITY CHECK
# ==============================================================================
import os
import sys
import shutil

print("=" * 70)
print("             SECTION B: REPOSITORY SETUP & INTEGRITY CHECK")
print("=" * 70)

# Configuration options for repository root
REPO_URL = ""  # Optional Git URL to clone (e.g. "https://github.com/your-username/i2-sdf.git")
REPO_DIR = "/content/i2-sdf"

if os.path.exists("main_recon.py") and os.path.exists("model"):
    PROJECT_ROOT = os.path.abspath(".")
    print(f"[INFO] Running directly within repository directory: {PROJECT_ROOT}")
elif REPO_URL and not os.path.exists(REPO_DIR):
    print(f"[INFO] Cloning repository from {REPO_URL} into {REPO_DIR} ...")
    !git clone {REPO_URL} {REPO_DIR}
    PROJECT_ROOT = REPO_DIR
    os.chdir(PROJECT_ROOT)
elif os.path.exists(REPO_DIR):
    PROJECT_ROOT = REPO_DIR
    os.chdir(PROJECT_ROOT)
    print(f"[INFO] Using existing repository at: {PROJECT_ROOT}")
else:
    PROJECT_ROOT = os.path.abspath(".")
    print(f"[INFO] Defaulting to current working directory: {PROJECT_ROOT}")

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"\n[STATUS] Working Directory: {os.getcwd()}")

# Verify required files
required_files = [
    "main_recon.py",
    "environment.yml",
    "config/synthetic.yml",
    "config/synthetic_physics.yml",
    "model/network/__init__.py",
    "model/trainer/recon.py",
    "test_grounding_poc.py",
    "data/normalize_cameras.py"
]

missing_files = []
for rel_path in required_files:
    full_path = os.path.join(PROJECT_ROOT, rel_path)
    exists = os.path.exists(full_path)
    status_str = "EXISTS" if exists else "MISSING"
    print(f"  • {rel_path:<35} : [{status_str}]")
    if not exists:
        missing_files.append(rel_path)

if missing_files:
    print(f"\n[WARNING] Missing files detected: {missing_files}")
else:
    print("\n[SUCCESS] All core repository files verified successfully!")

# Verify grounding constraint implementation in code
with open(os.path.join(PROJECT_ROOT, "model/network/__init__.py"), "r") as f:
    net_code = f.read()
    has_ground_loss = "get_ground_loss" in net_code and "ground_weight" in net_code

with open(os.path.join(PROJECT_ROOT, "model/trainer/recon.py"), "r") as f:
    train_code = f.read()
    has_ground_trainer = "ground_points" in train_code and "h_floor" in train_code

print(f"\n[INTEGRITY] Grounding Loss in model/network/__init__.py : {'VERIFIED' if has_ground_loss else 'MISSING'}")
print(f"[INTEGRITY] Grounding Logic in model/trainer/recon.py    : {'VERIFIED' if has_ground_trainer else 'MISSING'}")
print("=" * 70)


## Section C: Dependency Installation
Installs compatible versions of dependencies matching `environment.yml` for PyTorch Lightning, Open3D, Trimesh, PyMCubes, OpenCV (with OpenEXR), and Fast PyTorch KMeans.

In [ ]:
# ==============================================================================
# SECTION C: DEPENDENCY INSTALLATION
# ==============================================================================
import os
import sys

print("=" * 70)
print("             SECTION C: INSTALLING REQUIRED DEPENDENCIES")
print("=" * 70)

# System utilities
print("[INFO] Installing system utilities (ffmpeg, libopenexr-dev)...\n")
!apt-get update -qq && apt-get install -y -qq ffmpeg libopenexr-dev > /dev/null 2>&1

# Python packages pinned for I²-SDF and PyTorch Lightning compatibility
print("[INFO] Installing Python packages...\n")
!pip install -q \
    "pytorch-lightning==1.9.0" \
    "torchmetrics==0.11.4" \
    "open3d==0.17.0" \
    "trimesh==3.21.4" \
    "PyMCubes==0.1.4" \
    "fast-pytorch-kmeans==0.1.9" \
    "opencv-python==4.7.0.72" \
    "lpips==0.1.4" \
    "scikit-image==0.20.0" \
    "scikit-learn==1.2.2" \
    "scipy==1.9.1" \
    "pyyaml==6.0" \
    "rich==13.3.3" \
    "gputil==1.4.0" \
    "tensorboard==2.12.0" \
    "tensorboardx==2.6" \
    "tqdm==4.65.0"

print("\n[SUCCESS] All dependencies installed successfully.")
print("=" * 70)


## Section D: Dataset Setup (`bedroom_0` -> `scan0`)
Downloads the synthetic `bedroom_0.zip` dataset (~1.06 GB) from the official Kujiale CDN, extracts it into `data/synthetic/scan0/`, and generates camera normalization matrices.

In [ ]:
# ==============================================================================
# SECTION D: DATASET DOWNLOAD & SETUP (bedroom_0 -> scan0)
# ==============================================================================
import os
import sys
import zipfile
import shutil

print("=" * 70)
print("             SECTION D: DATASET SETUP (bedroom_0 / scan0)")
print("=" * 70)

DATASET_URL = "https://kloudsim-usa-cos.kujiale.com/interiorverse/i2-sdf/i2-sdf/bedroom_0.zip"
DATA_DIR = os.path.join(".", "data", "synthetic")
SCAN0_DIR = os.path.join(DATA_DIR, "scan0")
ZIP_PATH = os.path.join(DATA_DIR, "bedroom_0.zip")

os.makedirs(DATA_DIR, exist_ok=True)

# Download and extract if scan0 is not already present
if not os.path.exists(SCAN0_DIR) or not os.path.exists(os.path.join(SCAN0_DIR, "cameras.npz")):
    if not os.path.exists(ZIP_PATH):
        print(f"[INFO] Downloading synthetic dataset (bedroom_0.zip, ~1.06 GB)...")
        print(f"       Source: {DATASET_URL}")
        !curl -C - -o "{ZIP_PATH}" "{DATASET_URL}"
    
    print(f"\n[INFO] Extracting {ZIP_PATH} ...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(DATA_DIR)
    
    extracted_candidates = [d for d in os.listdir(DATA_DIR) if d.startswith("scan") and d != "scan0"]
    if extracted_candidates:
        src_scan = os.path.join(DATA_DIR, extracted_candidates[0])
        print(f"[INFO] Mapping extracted '{extracted_candidates[0]}' -> 'scan0' ...")
        if os.path.exists(SCAN0_DIR):
            shutil.rmtree(SCAN0_DIR)
        shutil.move(src_scan, SCAN0_DIR)
    elif os.path.exists(os.path.join(DATA_DIR, "bedroom_0")):
        shutil.move(os.path.join(DATA_DIR, "bedroom_0"), SCAN0_DIR)

print(f"\n[STATUS] Target scan0 directory: {os.path.abspath(SCAN0_DIR)}")

# Generate cameras_normalize.npz if needed
cam_norm_path = os.path.join(SCAN0_DIR, "cameras_normalize.npz")
if not os.path.exists(cam_norm_path):
    print("[INFO] Generating cameras_normalize.npz using data/normalize_cameras.py ...")
    !python data/normalize_cameras.py --id 0 -n data/synthetic -r 2.0
else:
    print("[STATUS] cameras_normalize.npz already exists.")

# Verify dataset modalities
expected_subdirs = ["image", "depth", "normal", "hdr", "mask", "val"]
print("\n[VERIFICATION] Dataset contents in data/synthetic/scan0:")
for subdir in expected_subdirs:
    sub_path = os.path.join(SCAN0_DIR, subdir)
    if os.path.exists(sub_path):
        count = len(os.listdir(sub_path))
        print(f"  • {subdir + '/':<15} : {count} files found")
    else:
        print(f"  • {subdir + '/':<15} : [MISSING]")

print(f"  • {'cameras.npz':<15} : {'EXISTS' if os.path.exists(os.path.join(SCAN0_DIR, 'cameras.npz')) else 'MISSING'}")
print(f"  • {'cameras_normalize.npz':<15} : {'EXISTS' if os.path.exists(os.path.join(SCAN0_DIR, 'cameras_normalize.npz')) else 'MISSING'}")
print("=" * 70)


## Section E: Environment & Modality Verification
Verifies Python imports, OpenEXR decoding via OpenCV, CUDA tensor ops, and YAML config parsing.

In [ ]:
# ==============================================================================
# SECTION E: ENVIRONMENT & MODALITY VERIFICATION
# ==============================================================================
import os
os.environ["OPENCV_IO_ENABLE_OPENEXR"] = "1"
import cv2
import yaml
import torch
import numpy as np
import pytorch_lightning as pl
import utils
import model
import dataset

print("=" * 70)
print("             SECTION E: ENVIRONMENT & MODALITY VERIFICATION")
print("=" * 70)

# 1. Test OpenEXR image reading via OpenCV
depth_sample_path = os.path.join("data", "synthetic", "scan0", "depth", "0000.exr")
if os.path.exists(depth_sample_path):
    depth_img = cv2.imread(depth_sample_path, -1)
    if depth_img is not None:
        print(f"[SUCCESS] OpenEXR Decoding: Loaded {depth_sample_path} (Shape: {depth_img.shape}, Dtype: {depth_img.dtype})")
    else:
        print(f"[ERROR] Failed to decode OpenEXR image!")
else:
    print(f"[WARNING] Sample depth EXR not found at {depth_sample_path}")

# 2. Test CUDA Tensor Allocation
x = torch.randn(1000, 1000, device="cuda")
y = torch.matmul(x, x)
print(f"[SUCCESS] PyTorch CUDA Matrix Ops: Result Norm = {y.norm().item():.4f}")

# 3. Test Config Loading
for conf_name in ["config/synthetic.yml", "config/synthetic_physics.yml"]:
    with open(conf_name, 'r') as f:
        cfg = utils.CfgNode(yaml.load(f, Loader=yaml.FullLoader))
    g_weight = getattr(cfg.loss, 'ground_weight', 0.0)
    print(f"[SUCCESS] Config Loaded: {conf_name:<30} | ground_weight = {g_weight}")

print("=" * 70)
print("[STATUS] All environment components verified successfully!")
print("=" * 70)


## Section F: Tiny Sanity Test (Optional Verification Only)
> **Note**: This section is an **optional verification check** and does **NOT** run automatically.
> Its purpose is strictly to verify that the Colab GPU runtime reproduces the existing proof-of-concept implementation (evaluating forward pass, loss values, non-zero SDF gradients, and parameter updates).
> Loss reduction over 20 iterations is an optimization sanity check, not a research benchmark result.

In [ ]:
# ==============================================================================
# SECTION F: TINY SANITY TEST (OPTIONAL VERIFICATION ONLY)
# ==============================================================================
# Set RUN_POC_TEST = True to execute the 20-iteration environment test.
RUN_POC_TEST = False

if not RUN_POC_TEST:
    print("[INFO] Section F (Sanity Test) is SKIPPED by default.")
    print("       To run this 20-iteration verification, set RUN_POC_TEST = True and re-run this cell.")
else:
    print("=" * 70)
    print("       RUNNING 20-ITERATION PROOF-OF-CONCEPT SANITY TEST")
    print("=" * 70)
    from test_grounding_poc import run_grounding_poc
    run_grounding_poc()


## Section G: Baseline Experiment (Standard I²-SDF, ground_weight = 0.0)
> **Safety Guard**: Full training does **NOT** launch automatically.
> You must explicitly set `RUN_BASELINE = True` to begin training.
> Before any long-running experiment, check the pre-flight inspection printout below.
>
> **Tip for Practical Runtime Calibration**: You can first perform a controlled short run (e.g. 1,000 steps) by adjusting `TRAIN_STEPS` to measure exact iterations-per-second on your Colab GPU before committing to the full 200,000-step training.

In [ ]:
# ==============================================================================
# SECTION G: BASELINE EXPERIMENT (Standard I²-SDF, ground_weight = 0.0)
# ==============================================================================
import os
import yaml
import torch
import utils

# Experiment Configuration
CONF_PATH = "config/synthetic.yml"
EXPNAME = "synthetic_baseline"
SCAN_ID = 0
GPU_DEVICE_ID = 0

# Target step budget (Default: 200,000 steps; adjust for calibration if desired)
TRAIN_STEPS = 200000

# Explicit confirmation toggle (Must be set to True by user to execute)
RUN_BASELINE = False

with open(CONF_PATH, 'r') as f:
    cfg = utils.CfgNode(yaml.load(f, Loader=yaml.FullLoader))
ground_weight = getattr(cfg.loss, 'ground_weight', 0.0)
expected_out_dir = os.path.abspath(os.path.join("exps", f"{EXPNAME}_{SCAN_ID}"))
gpu_name = torch.cuda.get_device_name(GPU_DEVICE_ID) if torch.cuda.is_available() else "N/A"
free_vram_gb = (torch.cuda.get_device_properties(GPU_DEVICE_ID).total_memory - torch.cuda.memory_allocated(GPU_DEVICE_ID)) / (1024**3)

print("=" * 75)
print("               BASELINE EXPERIMENT PRE-FLIGHT INSPECTION")
print("=" * 75)
print(f"• Configuration File       : {CONF_PATH}")
print(f"• Experiment Identifier    : {EXPNAME}")
print(f"• Scan ID                  : {SCAN_ID}")
print(f"• Grounding Loss Weight    : {ground_weight} (Baseline: Grounding Disabled)")
print(f"• Target Training Steps    : {TRAIN_STEPS}")
print(f"• Expected Output Directory: {expected_out_dir}")
print(f"• GPU Device in Use        : {gpu_name} (Free VRAM: {free_vram_gb:.2f} GB)")
print(f"• Execution Guard Status   : RUN_BASELINE = {RUN_BASELINE}")
print("=" * 75)

if not RUN_BASELINE:
    print("\n[ACTION REQUIRED] Baseline training was NOT launched.")
    print("To execute Baseline training, set `RUN_BASELINE = True` above and run this cell.")
    print(f"\nExact CLI command to be executed:")
    print(f"python main_recon.py --conf {CONF_PATH} --scan_id {SCAN_ID} -d {GPU_DEVICE_ID} --expname {EXPNAME}")
else:
    print(f"\n[INFO] Starting Baseline Training ({TRAIN_STEPS} steps)...\n")
    !python main_recon.py --conf {CONF_PATH} --scan_id {SCAN_ID} -d {GPU_DEVICE_ID} --expname {EXPNAME}


## Section H: Physics Experiment (Grounding-Loss I²-SDF, ground_weight = 0.1)
> **Safety Guard**: Full training does **NOT** launch automatically.
> You must explicitly set `RUN_PHYSICS = True` to begin training.
> The physics constraint applies differentiable downward SDF probe penalties to prevent floating structures.
>
> **Tip for Practical Runtime Calibration**: You can first perform a controlled short run (e.g. 1,000 steps) by adjusting `TRAIN_STEPS` to measure exact iterations-per-second on your Colab GPU before committing to the full 200,000-step training.

In [ ]:
# ==============================================================================
# SECTION H: PHYSICS EXPERIMENT (Grounding-Loss I²-SDF, ground_weight = 0.1)
# ==============================================================================
import os
import yaml
import torch
import utils

# Experiment Configuration
CONF_PATH = "config/synthetic_physics.yml"
EXPNAME = "synthetic_physics"
SCAN_ID = 0
GPU_DEVICE_ID = 0

# Target step budget (Default: 200,000 steps; adjust for calibration if desired)
TRAIN_STEPS = 200000

# Explicit confirmation toggle (Must be set to True by user to execute)
RUN_PHYSICS = False

with open(CONF_PATH, 'r') as f:
    cfg = utils.CfgNode(yaml.load(f, Loader=yaml.FullLoader))
ground_weight = getattr(cfg.loss, 'ground_weight', 0.1)
expected_out_dir = os.path.abspath(os.path.join("exps", f"{EXPNAME}_{SCAN_ID}"))
gpu_name = torch.cuda.get_device_name(GPU_DEVICE_ID) if torch.cuda.is_available() else "N/A"
free_vram_gb = (torch.cuda.get_device_properties(GPU_DEVICE_ID).total_memory - torch.cuda.memory_allocated(GPU_DEVICE_ID)) / (1024**3)

print("=" * 75)
print("               PHYSICS EXPERIMENT PRE-FLIGHT INSPECTION")
print("=" * 75)
print(f"• Configuration File       : {CONF_PATH}")
print(f"• Experiment Identifier    : {EXPNAME}")
print(f"• Scan ID                  : {SCAN_ID}")
print(f"• Grounding Loss Weight    : {ground_weight} (Physics: Grounding Active)")
print(f"• Target Training Steps    : {TRAIN_STEPS}")
print(f"• Expected Output Directory: {expected_out_dir}")
print(f"• GPU Device in Use        : {gpu_name} (Free VRAM: {free_vram_gb:.2f} GB)")
print(f"• Execution Guard Status   : RUN_PHYSICS = {RUN_PHYSICS}")
print("=" * 75)

if not RUN_PHYSICS:
    print("\n[ACTION REQUIRED] Physics experiment training was NOT launched.")
    print("To execute Physics training, set `RUN_PHYSICS = True` above and run this cell.")
    print(f"\nExact CLI command to be executed:")
    print(f"python main_recon.py --conf {CONF_PATH} --scan_id {SCAN_ID} -d {GPU_DEVICE_ID} --expname {EXPNAME}")
else:
    print(f"\n[INFO] Starting Physics-Guided Training ({TRAIN_STEPS} steps)...\n")
    !python main_recon.py --conf {CONF_PATH} --scan_id {SCAN_ID} -d {GPU_DEVICE_ID} --expname {EXPNAME}


## Section I: 3D Mesh Extraction (Marching Cubes)
Extracts surface geometry (`.ply` format) at resolution 512 from trained checkpoint SDF weights.

In [ ]:
# ==============================================================================
# SECTION I: MESH EXTRACTION (Marching Cubes)
# ==============================================================================
import os

RUN_MESH_EXTRACTION = False
RESOLUTION = 512  # Standard I²-SDF marching cube grid resolution

print("=" * 75)
print("                     SECTION I: 3D MESH EXTRACTION")
print("=" * 75)
print(f"• Marching Cubes Resolution : {RESOLUTION}")
print(f"• Execution Guard Status     : RUN_MESH_EXTRACTION = {RUN_MESH_EXTRACTION}")
print("=" * 75)

if not RUN_MESH_EXTRACTION:
    print("\n[ACTION REQUIRED] Mesh extraction is guarded.")
    print("To extract meshes from trained checkpoints, set `RUN_MESH_EXTRACTION = True` and re-run.")
    print("\nExact commands that will be executed:")
    print(f"1. Baseline Mesh: python main_recon.py --conf config/synthetic.yml --scan_id 0 -d 0 --expname synthetic_baseline --test --test_mode mesh --resolution {RESOLUTION}")
    print(f"2. Physics Mesh : python main_recon.py --conf config/synthetic_physics.yml --scan_id 0 -d 0 --expname synthetic_physics --test --test_mode mesh --resolution {RESOLUTION}")
else:
    print("\n[1/2] Extracting Baseline 3D Mesh...")
    !python main_recon.py --conf config/synthetic.yml --scan_id 0 -d 0 --expname synthetic_baseline --test --test_mode mesh --resolution {RESOLUTION}
    
    print("\n[2/2] Extracting Physics-Guided 3D Mesh...")
    !python main_recon.py --conf config/synthetic_physics.yml --scan_id 0 -d 0 --expname synthetic_physics --test --test_mode mesh --resolution {RESOLUTION}
    
    print("\n[SUCCESS] Mesh extraction complete. Output .ply files saved in exps directories.")


## Section J: Quantitative Comparison & Grounding Metrics
Evaluates mesh floor contact distributions and compares visual metrics (PSNR, SSIM, LPIPS).

In [ ]:
# ==============================================================================
# SECTION J: QUANTITATIVE COMPARISON & GROUNDING METRICS
# ==============================================================================
import os
import glob
import torch
import trimesh
import numpy as np

print("=" * 75)
print("             SECTION J: QUANTITATIVE & STRUCTURAL COMPARISON")
print("=" * 75)

def analyze_mesh_grounding(mesh_path, label="Model"):
    if not os.path.exists(mesh_path):
        print(f"[{label}] Mesh file not found at: {mesh_path}")
        return None
    
    mesh = trimesh.load(mesh_path)
    vertices = mesh.vertices
    z_coords = vertices[:, 2]
    
    return {
        'label': label,
        'num_vertices': len(vertices),
        'num_faces': len(mesh.faces),
        'min_z': float(np.min(z_coords)),
        'p1_z': float(np.percentile(z_coords, 1)),
        'p5_z': float(np.percentile(z_coords, 5)),
        'mean_z': float(np.mean(z_coords)),
        'max_z': float(np.max(z_coords)),
    }

baseline_mesh_files = glob.glob("exps/synthetic_baseline_0/**/*.ply", recursive=True)
physics_mesh_files = glob.glob("exps/synthetic_physics_0/**/*.ply", recursive=True)

print(f"[INFO] Found {len(baseline_mesh_files)} Baseline mesh(es)")
print(f"[INFO] Found {len(physics_mesh_files)} Physics mesh(es)")

if baseline_mesh_files and physics_mesh_files:
    b_stats = analyze_mesh_grounding(baseline_mesh_files[0], "Baseline (Standard I²-SDF)")
    p_stats = analyze_mesh_grounding(physics_mesh_files[0], "Physics (Grounding-Guided)")
    
    print("\n" + "-" * 75)
    print(f"{'Metric':<30} | {'Baseline (Unconstrained)':<22} | {'Physics (Grounding-Guided)':<22}")
    print("-" * 75)
    print(f"{'Vertex Count':<30} | {b_stats['num_vertices']:<22} | {p_stats['num_vertices']:<22}")
    print(f"{'Face Count':<30} | {b_stats['num_faces']:<22} | {p_stats['num_faces']:<22}")
    print(f"{'Min Height (min z)':<30} | {b_stats['min_z']:<22.4f} | {p_stats['min_z']:<22.4f}")
    print(f"{'1st Percentile Height (p1)':<30} | {b_stats['p1_z']:<22.4f} | {p_stats['p1_z']:<22.4f}")
    print(f"{'5th Percentile Height (p5)':<30} | {b_stats['p5_z']:<22.4f} | {p_stats['p5_z']:<22.4f}")
    print("-" * 75)
else:
    print("[INFO] Quantitative comparison will display full results once checkpoints and meshes are generated.")
print("=" * 75)


## Section K: Persistent Storage (Google Drive Backup)
Mounts Google Drive and copies all checkpoints, extracted meshes, rendering plots, and TensorBoard logs for permanent storage.

In [ ]:
# ==============================================================================
# SECTION K: PERSISTENT STORAGE (GOOGLE DRIVE BACKUP)
# ==============================================================================
import os
import shutil

BACKUP_TO_GDRIVE = False
GDRIVE_DEST_DIR = "/content/drive/MyDrive/i2sdf_experiments"

print("=" * 75)
print("             SECTION K: GOOGLE DRIVE PERSISTENT BACKUP")
print("=" * 75)
print(f"• Backup Destination       : {GDRIVE_DEST_DIR}")
print(f"• Execution Guard Status   : BACKUP_TO_GDRIVE = {BACKUP_TO_GDRIVE}")
print("=" * 75)

if not BACKUP_TO_GDRIVE:
    print("\n[INFO] Google Drive backup is currently idle.")
    print("To mount Drive and copy checkpoints/meshes, set `BACKUP_TO_GDRIVE = True` and run this cell.")
else:
    try:
        from google.colab import drive
        print("[INFO] Mounting Google Drive at /content/drive ...")
        drive.mount('/content/drive')
        
        os.makedirs(GDRIVE_DEST_DIR, exist_ok=True)
        src_exps = os.path.abspath("exps")
        if os.path.exists(src_exps):
            print(f"[INFO] Copying {src_exps} -> {GDRIVE_DEST_DIR} ...")
            !cp -r exps/* "{GDRIVE_DEST_DIR}/"
            print(f"[SUCCESS] All checkpoints, meshes, and logs backed up to {GDRIVE_DEST_DIR}")
        else:
            print(f"[WARNING] No local 'exps' directory found to back up.")
    except Exception as e:
        print(f"[ERROR] Google Drive backup failed: {e}")


## Summary & Exact Command Reference
Printout of the exact commands for all operations.

In [ ]:
# ==============================================================================
# SUMMARY: EXACT COMMAND CHEATSHEET
# ==============================================================================
print("=" * 80)
print("                I²-SDF REPRODUCIBILITY COMMAND CHEATSHEET")
print("=" * 80)
print("\n1. BASELINE TRAINING (Standard I²-SDF, ground_weight=0.0):")
print("   python main_recon.py --conf config/synthetic.yml --scan_id 0 -d 0 --expname synthetic_baseline")
print("\n2. PHYSICS EXPERIMENT TRAINING (Physics-Guided Grounding, ground_weight=0.1):")
print("   python main_recon.py --conf config/synthetic_physics.yml --scan_id 0 -d 0 --expname synthetic_physics")
print("\n3. BASELINE MESH EXTRACTION (Marching Cubes at Resolution 512):")
print("   python main_recon.py --conf config/synthetic.yml --scan_id 0 -d 0 --expname synthetic_baseline --test --test_mode mesh --resolution 512")
print("\n4. PHYSICS MESH EXTRACTION (Marching Cubes at Resolution 512):")
print("   python main_recon.py --conf config/synthetic_physics.yml --scan_id 0 -d 0 --expname synthetic_physics --test --test_mode mesh --resolution 512")
print("\n5. NOVEL VIEW SYNTHESIS EVALUATION (Validation Views):")
print("   python main_recon.py --conf config/synthetic.yml --scan_id 0 -d 0 --expname synthetic_baseline --test --is_val")
print("   python main_recon.py --conf config/synthetic_physics.yml --scan_id 0 -d 0 --expname synthetic_physics --test --is_val")
print("=" * 80)
